# Architecture Defense

Compare retained decoder state without conflating distinct mechanisms.

This is the full **PyTorch** version of a challenge from [LLM Quest](https://bankoti.github.io/llm-quest). The in-browser challenge grades numpy; here you work with real tensors. Fill in each `TODO`, then run the checks cell.

Runs on the free Colab CPU runtime; switch to a GPU via *Runtime > Change runtime type* if you want to experiment at scale. PyTorch comes preinstalled.

In [ ]:
def kv_cache_bytes(
    *,
    layers: int,
    tokens: int,
    kv_heads: int,
    head_dim: int,
    bytes_per_scalar: int = 2,
) -> int:
    return 2 * layers * tokens * kv_heads * head_dim * bytes_per_scalar


def compressed_state_bytes(
    *, layers: int, tokens: int, cached_width: int, bytes_per_scalar: int = 2
) -> int:
    """Budget a generic per-token latent; this is not an MLA implementation."""
    return layers * tokens * cached_width * bytes_per_scalar


base = {"layers": 32, "tokens": 8192, "head_dim": 128}
mha = kv_cache_bytes(**base, kv_heads=32)
gqa = kv_cache_bytes(**base, kv_heads=8)
mqa = kv_cache_bytes(**base, kv_heads=1)
generic_compressed = compressed_state_bytes(
    layers=base["layers"], tokens=base["tokens"], cached_width=512
)

In [ ]:
assert mha // gqa == 4
assert gqa // mqa == 8
assert generic_compressed == mqa * 2

print(
    "Architecture lab 06 passed",
    {
        "mha_gib": mha / 2**30,
        "gqa_gib": gqa / 2**30,
        "mqa_gib": mqa / 2**30,
        "generic_compressed_gib": generic_compressed / 2**30,
    },
)